In [1]:
import pandas as pd
import datetime
import sqlite3
import pymysql
import pandas.io.sql as psql
from datetime import datetime as dt
import numpy as np
import pandas.tseries.offsets as offsets
import sqlalchemy as sqa
import matplotlib.pyplot as plt
import python_ss as ps
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import json

import os
from decimal import Decimal
import calendar
import utils
from utils import *
#importlib.reload(utils)
print(os.getcwd())


import os
import ast
import db_dtypes
from google.cloud import bigquery
from google.oauth2 import service_account
from google.cloud import secretmanager

C:\Users\suehara\Desktop\お転機BOX\ぱいそん練習


In [2]:
#!/usr/bin/env python
# coding: utf-8

import os
import ast
import json
# importでエラーが出てしまった場合は、コマンドプロンプトにて「pip install ”必要なモジュール”」でインストールしていただく必要がございます。
# 例. pip install db_dtypes
import pandas as pd
import db_dtypes
from google.cloud import bigquery
from google.oauth2 import service_account
from google.cloud import secretmanager

def access_secret_version(project_id, secret_id, version_id='latest'):
    client = secretmanager.SecretManagerServiceClient()

    name = f"projects/{project_id}/secrets/{secret_id}/versions/{version_id}"
    response = client.access_secret_version(request={"name": name})
    payload = response.payload.data.decode("UTF-8")
    return ast.literal_eval(payload)

# 上記関数を実行するコードが記載されています。こちらもそのままお使いください。
credentials = service_account.Credentials.from_service_account_info(
  access_secret_version('temp-for-sandbox', 'TEMP_CREDENTIAL_KEY'),
  scopes=["https://www.googleapis.com/auth/cloud-platform"],
)


C:\Users\suehara\Anaconda3\lib\site-packages\google\auth\_default.py:78: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [3]:
def access_secret_version(project_id, secret_id, version_id='latest'):
    client = secretmanager.SecretManagerServiceClient()

    name = f"projects/{project_id}/secrets/{secret_id}/versions/{version_id}"
    response = client.access_secret_version(request={"name": name})
    payload = response.payload.data.decode("UTF-8")
    return ast.literal_eval(payload)

In [4]:
# 上記関数を実行するコードが記載されています。こちらもそのままお使いください。
credentials = service_account.Credentials.from_service_account_info(
  access_secret_version('temp-for-sandbox', 'TEMP_CREDENTIAL_KEY'),
  scopes=["https://www.googleapis.com/auth/cloud-platform"],)

In [5]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

In [6]:
#ロンザンのマスタデータのパスの設定
path1 = r"\\172.16.0.232\CoffeeCrazy\経営ソリューション事業部\□シニアスカウト事業部□\01 全体進捗\02 行動カレンダー\pythonデータ"

#マスタデータを読み込み
# master1= 日付・月・カレンダー週・Qデータ(2019/10/1	23-10月	9月5W(23日～1日)	23-1Q)
master1 = pd.read_excel(path1 + "\※最新※ロンザン社長資料マスタデータ.xlsx",sheet_name='マスタ',usecols=[0,1,2,3])

# master2= 在籍Q・人マスタ・略・user_id・所属フラグ・ロンザン所属フラグ(23-2Q	五十嵐奏子	五十嵐　igarashi ミドル	0)
master2 = pd.read_excel(path1 + "\※最新※ロンザン社長資料マスタデータ.xlsx",sheet_name='マスタ',usecols=[6,7,8,9,10,11])

# master3= 人マスタ・略・チーム・レイヤー(大仲研司	大仲	1課	部責)
# master3 = pd.read_excel(path1 + "\※最新※ロンザン社長資料マスタデータ.xlsx",sheet_name='マスタ',usecols=[14,15,16,17])

# master4= ヨミ表選択・丸め(人事部	人事部紹介)
master4 = pd.read_excel(path1 + "\※最新※ロンザン社長資料マスタデータ.xlsx",sheet_name='マスタ',usecols=[20,21])

# master5= 計上Q・修正後ポイント・Q計上時ポイント・掛け率(19-3Q	5,712	7,297	78%)
master5 = pd.read_excel(path1 + "\※最新※ロンザン社長資料マスタデータ.xlsx",sheet_name='マスタ',usecols=[23,24,25,26])

#master3 = master3.rename(columns={"人マスタ.1": "人マスタ","sei_plus.1":"sei_plus"}) #カラム名変更
#master3 = master3.dropna(subset=['人マスタ', 'sei_plus'])

master2 = master2.dropna(subset=['人マスタ', 'sei_plus'])

master4 = master4.dropna(subset=['ヨミ表選択'])

master5 = master5.rename(columns={"計上Q.1": "計上Q","掛け率.1":"掛け率"}) #カラム名変更
master5 = master5.dropna(subset=['計上Q'])

In [7]:
#日付のマスタデータのパスの設定
path2 = r"\\172.16.0.232\CoffeeCrazy\総合市場開発部\50　個人フォルダ\40　【大阪】\塩澤\マスタ"
Q_master = pd.read_excel(path2 + "\Qマスタ.xlsx",usecols=[0,1,2,3,5,8,11,12,13])
Q_master["日付"] = pd.to_datetime(Q_master["日付"], format='%Y/%m/%d') #日付データを変換
Q_master.columns

Index(['日付', 'Q', '月', 'Q同営', 'Q同旬', '営業日比較', '同営業日比較', '同旬月日比較', '期間対象外'], dtype='object')

In [8]:
Q_master = Q_master[["日付","Q","月"]]
Q_master = Q_master.rename(columns={"日付":"toroku_date"})

In [9]:
Q_master['toroku_date'] = Q_master['toroku_date'].dt.strftime('%Y/%m/%d')
Q_master['月'] = Q_master['月'].dt.strftime('%Y/%m')

In [10]:
Q_master

,toroku_date,Q,月
0,2009/02/02,12-2Q,2009/02
1,2009/02/03,12-2Q,2009/02
2,2009/02/04,12-2Q,2009/02
3,2009/02/05,12-2Q,2009/02
4,2009/02/06,12-2Q,2009/02
...,...,...,...
5353,2023/09/30,26-4Q,2023/09
5354,2023/10/01,26-4Q,2023/10
5355,2023/10/02,26-4Q,2023/10
5356,2023/10/03,26-4Q,2023/10


## 登録情報更新

In [11]:
tktoroku_query = """
select
	inf.id as ID,
	format_date('%Y/%m/%d',inf.created_at) as toroku_date,
	concat(trim(replace(inf.shi,'　',' ')),trim(replace(inf.mei,'　',' '))) as name,
	case when inf.income is null then "0"
	     when inf.income = "300万円未満" then "200"
			 when inf.income = "300～399万円" then "300"
			 when inf.income = "400～499万円" then "400"
			 when inf.income = "500～599万円" then "500"
			 when inf.income = "600～699万円" then "600"
			 when inf.income = "700～799万円" then "700"
			 when inf.income = "800～999万円" then "800"
			 when inf.income = "800～899万円" then "800"
			 when inf.income = "900～999万円" then "900"
			 when inf.income = "1000～1099万円" then "1000"
			 when inf.income = "1100～1199万円" then "1100"
			 when inf.income = "1200～1299万円" then "1200"
			 when inf.income = "1300～1399万円" then "1300"
			 when inf.income = "1400～1499万円" then "1400"
			 when inf.income = "1500～1599万円" then "1500"
			 when inf.income = "1600～1699万円" then "1600"
			 when inf.income = "1700～1799万円" then "1700"
			 when inf.income = "1800～1899万円" then "1800"
			 when inf.income = "1900～1999万円" then "1900"
			 when inf.income = "2000万円以上" then "2000"
			 else "" end as income
from `temp-380708.live_tenki.user_info` inf
where inf.id >= 46273
order by id
"""

client = bigquery.Client(credentials=credentials, project=credentials.project_id)
tktoroku = client.query(tktoroku_query).result().to_dataframe()

tktoroku["income"] = tktoroku["income"].str.replace('None', '')
tktoroku["income"] = tktoroku["income"].astype("int64")
tktoroku["ID"] = tktoroku["ID"].astype(str)
tktoroku["toroku_date"] = tktoroku["toroku_date"].astype(str)

In [12]:
#2020年1月以降の手上げ情報取得
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"C:\Users\suehara\Desktop\お転機BOX\ぱいそん練習\python_ss\credentials.json"
service = ps.get_auth(SCOPES,json_path)
SPREADSHEET_ID = '15qRR-yfJxgCh_TpXwBjBHNAnc8yAeTUCF0-Y_N6GoIo'
Sheet_NAME = '候補者状況!A'
Sheet_row = ":O"
RANGE_NAME = Sheet_NAME+Sheet_row
teageinfo = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)
teageinfo = teageinfo[["ID","登録日時","処理フラグ","フリ先","手あげ日付","担当","手あげ"]]

teageinfo["ID"] = teageinfo["ID"].fillna(0)

#2019年9月以降の手上げ情報取得
SPREADSHEET_ID = '1fOGhqvCoER3YYv2npDUT1KAB6Fw9fYfQ29Vf52p5Nts'
Sheet_NAME = '～19.12.31!A'
Sheet_row = ":O"
RANGE_NAME = Sheet_NAME+Sheet_row
pastteageinfo = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)
pastteageinfo = pastteageinfo[["ID","登録日時","処理フラグ","フリ先","手あげ日付","担当","手あげ"]]

pastteageinfo["ID"] = pastteageinfo["ID"].fillna(0)
teageinfo = pd.concat([pastteageinfo,teageinfo],axis=0)
teageinfo["ID"] = teageinfo["ID"].astype(str)
teageinfoA = teageinfo.copy()

In [13]:
tktoroku = pd.merge(tktoroku,teageinfoA,how="left",on=("ID"))

tktoroku = tktoroku.rename(columns={"フリ先":"事業部","処理フラグ":"結果","手あげ日付":"teage_date"})
tktoroku = tktoroku[["ID","toroku_date","name","income","結果","事業部","teage_date","担当","手あげ"]]

tktoroku = tktoroku[tktoroku["toroku_date"] >= "2019/10/02"]

tktoroku['income_below700'] = tktoroku['income'] < 700
tktoroku['income_between700and1000'] = (tktoroku['income'] >= 700) & (tktoroku['income'] < 1000)
tktoroku['income_over1000'] = tktoroku['income'] >= 1000
tktoroku

,ID,toroku_date,name,income,結果,事業部,teage_date,担当,手あげ,income_below700,income_between700and1000,income_over1000
49,46322,None,None,0,None,None,None,None,None,True,False,False
52,46325,None,None,0,None,None,None,None,None,True,False,False
53,46326,None,None,0,None,None,None,None,None,True,False,False
56,46329,2019/10/02,佐川豪,800,,レイノス,2019/10/02,青田 翔,青田 翔,False,True,False
57,46330,2019/10/02,小関一浩,800,None,None,None,None,None,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...
62789,109064,2023/09/15,大森正,800,None,None,None,None,None,False,True,False
62790,109065,2023/09/15,藤原功,200,None,None,None,None,None,True,False,False
62791,109066,2023/09/15,広岡一朗,1300,None,None,None,None,None,False,False,True
62792,109067,2023/09/15,橋詰秀樹,600,NaN,NaN,NaN,NaN,NaN,True,False,False


In [14]:
Q_master1 = Q_master.copy()

#登録データを作成
toroku = pd.merge(tktoroku,Q_master1,how="left",on="toroku_date")
toroku["type"] = "登録数"

df1 = toroku.pivot_table(index=["type"],columns="Q",aggfunc="count",values="toroku_date").fillna(0)
df2 = toroku.pivot_table(index=["type"],columns="月",aggfunc="count",values="toroku_date").fillna(0)
toroku_all=  pd.concat([df1,df2],axis=1).reset_index()

In [15]:
df1 = toroku.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_below700").fillna(0)
df2 = toroku.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_below700").fillna(0)
below_toroku =  pd.concat([df1,df2],axis=1).reset_index()

df1 = toroku.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_between700and1000").fillna(0)
df2 = toroku.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_between700and1000").fillna(0)
between_toroku =  pd.concat([df1,df2],axis=1).reset_index()

df1 = toroku.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_over1000").fillna(0)
df2 = toroku.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_over1000").fillna(0)
over_toroku =  pd.concat([df1,df2],axis=1).reset_index()

income_toroku_all=  pd.concat([below_toroku,between_toroku,over_toroku],axis=0).reset_index()

In [16]:
Q_master2 = Q_master.rename(columns={"toroku_date":"teage_date"})
teage = pd.merge(tktoroku,Q_master2,how="left",on="teage_date")
teage["type"] = "手上げ数"

df1 = teage.pivot_table(index=["type"],columns="Q",aggfunc="count",values="teage_date").fillna(0)
df2 = teage.pivot_table(index=["type"],columns="月",aggfunc="count",values="teage_date").fillna(0)
teage_all=  pd.concat([df1,df2],axis=1).reset_index()

In [17]:
df1 = teage.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_below700").fillna(0)
df2 = teage.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_below700").fillna(0)
below_teage =  pd.concat([df1,df2],axis=1).reset_index()

df1 = teage.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_between700and1000").fillna(0)
df2 = teage.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_between700and1000").fillna(0)
between_teage =  pd.concat([df1,df2],axis=1).reset_index()

df1 = teage.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_over1000").fillna(0)
df2 = teage.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_over1000").fillna(0)
over_teage =  pd.concat([df1,df2],axis=1).reset_index()

income_teage_all=  pd.concat([below_teage,between_teage,over_teage],axis=0).reset_index()

In [18]:
income_teage_all

,index,type,22-2Q,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,2019/01,2019/10,2019/11,2019/12,2020/01,2020/02,2020/03,2020/04,2020/05,2020/06,2020/07,2020/08,2020/09,2020/10,2020/11,2020/12,2021/01,2021/02,2021/03,2021/04,2021/05,2021/06,2021/07,2021/08,2021/09,2021/10,2021/11,2021/12,2022/01,2022/02,2022/03,2022/04,2022/05,2022/06,2022/07,2022/08,2022/09,2022/10,2022/11,2022/12,2023/01,2023/02,2023/03,2023/04,2023/05,2023/06,2023/07,2023/08,2023/09
0,0,手上げ数,0,376,331,465,451,528,452,463,391,210,209,146,172,137,126,240,255,0,133,143,108,132,109,88,144,158,162,216,114,116,166,155,207,146,126,180,139,163,161,137,162,87,75,91,53,68,89,49,47,33,68,54,68,47,14,33,93,79,24,20,28,112,105,96,108,46
1,0,手上げ数,1,477,500,756,703,834,918,740,664,345,465,429,458,169,277,475,460,1,157,198,126,198,155,148,185,271,310,286,190,212,307,297,230,176,173,569,222,248,270,236,263,149,138,136,94,159,190,116,111,129,184,186,145,125,21,49,103,124,79,76,105,176,195,189,181,83
2,0,手上げ数,0,919,1014,1232,1181,1114,1233,1045,1195,876,1084,773,863,406,815,1034,1089,0,288,366,269,361,332,321,340,423,485,508,320,333,394,385,335,335,383,515,300,322,423,403,409,345,309,329,291,362,392,328,203,245,320,362,303,190,52,99,265,294,243,282,256,384,407,395,444,223


In [19]:
joryu = pd.concat([toroku_all,teage_all],axis=0).reset_index()
joryu = joryu.drop(['22-2Q','2019/01'], axis=1)

income_joryu = pd.concat([income_toroku_all,income_teage_all],axis=0).reset_index()
income_joryu = income_joryu.drop(['22-2Q','2019/01'], axis=1)

In [20]:
income_joryu

,level_0,index,type,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,2019/10,2019/11,2019/12,2020/01,2020/02,2020/03,2020/04,2020/05,2020/06,2020/07,2020/08,2020/09,2020/10,2020/11,2020/12,2021/01,2021/02,2021/03,2021/04,2021/05,2021/06,2021/07,2021/08,2021/09,2021/10,2021/11,2021/12,2022/01,2022/02,2022/03,2022/04,2022/05,2022/06,2022/07,2022/08,2022/09,2022/10,2022/11,2022/12,2023/01,2023/02,2023/03,2023/04,2023/05,2023/06,2023/07,2023/08,2023/09
0,0,0,登録数,1383,1396,1799,1913,1712,1699,1370,1624,1324,1256,1322,1046,426,705,964,773,450,573,414,462,501,434,523,602,749,688,534,561,623,484,643,568,615,478,349,525,496,425,549,596,516,478,395,466,422,380,454,348,517,416,378,232,38,79,324,231,187,288,317,288,358,333,288,137
1,1,0,登録数,809,942,1026,1147,1002,917,806,1043,886,1059,1009,865,259,472,699,630,275,344,225,315,334,302,281,353,447,416,298,334,367,345,307,279,298,323,231,279,296,277,334,384,348,297,304,404,352,303,312,286,414,362,288,197,33,63,172,180,136,159,215,201,281,259,252,109
2,2,0,登録数,1133,1284,1361,1517,1171,1314,1097,1443,1243,1511,1335,1127,491,1031,1160,1258,374,453,350,436,451,400,390,456,552,686,367,380,419,397,377,392,446,454,308,347,442,428,455,511,458,398,451,552,492,466,418,396,524,505,397,208,47,132,331,362,304,366,376,319,465,440,526,272
3,0,0,手上げ数,376,331,465,451,528,452,463,391,210,209,146,172,137,126,240,255,133,143,108,132,109,88,144,158,162,216,114,116,166,155,207,146,126,180,139,163,161,137,162,87,75,91,53,68,89,49,47,33,68,54,68,47,14,33,93,79,24,20,28,112,105,96,108,46
4,1,0,手上げ数,477,500,756,703,834,918,740,664,345,465,429,458,169,277,475,460,157,198,126,198,155,148,185,271,310,286,190,212,307,297,230,176,173,569,222,248,270,236,263,149,138,136,94,159,190,116,111,129,184,186,145,125,21,49,103,124,79,76,105,176,195,189,181,83
5,2,0,手上げ数,919,1014,1232,1181,1114,1233,1045,1195,876,1084,773,863,406,815,1034,1089,288,366,269,361,332,321,340,423,485,508,320,333,394,385,335,335,383,515,300,322,423,403,409,345,309,329,291,362,392,328,203,245,320,362,303,190,52,99,265,294,243,282,256,384,407,395,444,223



tktoroku.replace([np.inf, -np.inf], np.nan, inplace=True)
tktoroku.fillna('', inplace=True)

tktoroku = tktoroku.values.tolist()

#スプレッドシートに記載する
SPREADSHEET_ID = '1ofer5LQADQ9ppKrtQWWbOyhyEmcrBx_c_O4UmVFe3wU'
Sheet_NAME = '登録!A'
Sheet_row = "2"
RANGE_NAME = Sheet_NAME+Sheet_row
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,tktoroku,service)

## 初期交渉

In [21]:
rzshokikosho_query = """
SELECT
  shk.kohosha_id,
  shk.tenki_id,
  consts.name as ap_source,
  syi2.sei_plus as ap_kakutokusha,
  case when syi1.sei_plus is not null then syi1.sei_plus
  else shk.mendan_tanto end as mendan_tanto,
  format_date('%Y/%m/%d',shk.kosho_setteibi) as kosho_setteibi,
  format_date('%Y/%m/%d',shk.kosho_yoteibi) as kosho_yoteibi,
  format_date('%Y/%m/%d',shk.kosho_jisshibi) as kosho_jisshibi,
  case when shk.kosho_yoteibi >= CURRENT_DATE("Asia/Tokyo") then '実施前'
       when shk.jisshi_flag=2 and shk.deleted=2 then "CXL"
       when shk.jisshi_flag=0 and shk.deleted=0 and shk.nittei_chosei=1 then "日程調整中"
       when shk.jisshi_flag=1 then '実施'
       when shk.jisshi_flag=0 then '未報告'
       else cast(shk.jisshi_flag as string) end as jisshi,
  khs.birth_year,
  khs.annual_income as income,
  shk.kosho_seq,
  shk.cxl_riyu
  FROM `temp-380708.live_rhs.shokikoshos` shk
  left join `temp-380708.live_rhs.kohoshas` khs on shk.kohosha_id = khs.id
  left join `temp-380708.live_company.syain` syi1 on shk.mendan_tanto = syi1.user_id
  left join `temp-380708.live_company.syain` syi2 on shk.ap_kakutoku = syi2.user_id
  left join (SELECT 
              code,
              name
             FROM `temp-380708.live_rhs.sys_consts`
             where group_code = 19) consts on shk.ap_source = consts.code
  where shk.kosho_setteibi >= date '2019-10-02'
  and consts.name = '転機社長名鑑'
  and (shk.saikosho_kaisu < 1 or shk.saikosho_kaisu >= 900)
  order by shk.kosho_setteibi
"""

client = bigquery.Client(credentials=credentials, project=credentials.project_id)
shokikosho = client.query(rzshokikosho_query).result().to_dataframe()

shokikosho['income_below700'] = shokikosho['income'] < 700
shokikosho['income_between700and1000'] = (shokikosho['income'] >= 700) & (shokikosho['income'] < 1000)
shokikosho['income_over1000'] = shokikosho['income'] >= 1000

shoki_settei = shokikosho.copy()
shoki_jisshi = shokikosho.copy()

shokikosho

,kohosha_id,tenki_id,ap_source,ap_kakutokusha,mendan_tanto,kosho_setteibi,kosho_yoteibi,kosho_jisshibi,jisshi,birth_year,income,kosho_seq,cxl_riyu,income_below700,income_between700and1000,income_over1000
0,15040,<NA>,転機社長名鑑,内藤ち,長崎文,2019/10/02,2019/10/08,None,CXL,<NA>,NaN,1,CXL,False,False,False
1,15049,<NA>,転機社長名鑑,内藤ち,増田智,2019/10/02,2019/10/23,None,日程調整中,<NA>,NaN,1,None,False,False,False
2,15052,<NA>,転機社長名鑑,内藤ち,入学琴,2019/10/02,2019/10/07,2019/10/07,実施,1965,900.0,1,None,False,True,False
3,15056,<NA>,転機社長名鑑,内藤ち,大谷昌,2019/10/02,2019/10/07,2019/10/07,実施,1964,1100.0,1,None,False,False,True
4,15045,<NA>,転機社長名鑑,内藤ち,仙頭克,2019/10/02,2019/10/07,2019/10/07,実施,1965,2200.0,1,None,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18376,47002,108993,転機社長名鑑,内藤ち,松田和,2023/09/14,2023/09/21,None,実施前,1969,NaN,1,None,False,False,False
18377,47003,109012,転機社長名鑑,内藤ち,原口虎,2023/09/14,2023/09/25,None,実施前,1976,NaN,1,None,False,False,False
18378,47015,108997,転機社長名鑑,内藤ち,原口虎,2023/09/14,2023/09/22,None,実施前,1970,NaN,1,None,False,False,False
18379,47008,109023,転機社長名鑑,内藤ち,谷地佳,2023/09/14,2023/09/19,None,実施前,<NA>,NaN,1,None,False,False,False


In [22]:
#設定テーブルを作成
Q_master3 = Q_master.rename(columns={"toroku_date":"kosho_setteibi"})
shoki_settei = pd.merge(shoki_settei,Q_master3,on=("kosho_setteibi"),how=("left"))

#再交渉テーブルを作成
shoki_sai_settei = shoki_settei[shoki_settei['kosho_seq'] >1]
shoki_sai_settei["type"] = "再交渉設定数"

#初期交渉テーブルを作成
shoki_settei = shoki_settei[shoki_settei['kosho_seq'] <= 1]
shoki_settei["type"] = "初期交渉設定数"

#実施テーブルを作成
Q_master4 = Q_master.rename(columns={"toroku_date":"kosho_jisshibi"})
shoki_jisshi = pd.merge(shoki_jisshi,Q_master4,on=("kosho_jisshibi"),how=("left"))

#再交渉実施テーブルを作成
shoki_sai_jisshi = shoki_jisshi[shoki_jisshi['kosho_seq'] >1]
shoki_sai_jisshi["type"] = "再交渉実施数"

#初期交渉実施テーブルを作成
shoki_jisshi = shoki_jisshi[shoki_jisshi['kosho_seq'] <= 1]
shoki_jisshi["type"] = "初期交渉実施数"

C:\Users\suehara\AppData\Local\Temp\ipykernel_18940\123396387.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  shoki_sai_settei["type"] = "再交渉設定数"
C:\Users\suehara\AppData\Local\Temp\ipykernel_18940\123396387.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  shoki_sai_jisshi["type"] = "再交渉実施数"


In [23]:
#初期交渉設定データ
df1 = shoki_settei.pivot_table(index=["type"],columns="Q",aggfunc="count",values="kosho_setteibi").fillna(0)
df2 = shoki_settei.pivot_table(index=["type"],columns="月",aggfunc="count",values="kosho_setteibi").fillna(0)
shoki_settei_all=  pd.concat([df1,df2],axis=1).reset_index()
#再交渉設定データ
df1 = shoki_sai_settei.pivot_table(index=["type"],columns="Q",aggfunc="count",values="kosho_setteibi").fillna(0)
df2 = shoki_sai_settei.pivot_table(index=["type"],columns="月",aggfunc="count",values="kosho_setteibi").fillna(0)
shoki_sai_settei_all=  pd.concat([df1,df2],axis=1).reset_index()
#初期交渉実施データ
df1 = shoki_jisshi.pivot_table(index=["type"],columns="Q",aggfunc="count",values="kosho_setteibi").fillna(0)
df2 = shoki_jisshi.pivot_table(index=["type"],columns="月",aggfunc="count",values="kosho_setteibi").fillna(0)
shoki_jisshi_all=  pd.concat([df1,df2],axis=1).reset_index()
#再交渉実施データ
df1 = shoki_sai_jisshi.pivot_table(index=["type"],columns="Q",aggfunc="count",values="kosho_setteibi").fillna(0)
df2 = shoki_sai_jisshi.pivot_table(index=["type"],columns="月",aggfunc="count",values="kosho_setteibi").fillna(0)
shoki_sai_jisshi_all =  pd.concat([df1,df2],axis=1).reset_index()

shokikosho_all = pd.concat([shoki_settei_all,shoki_jisshi_all],axis=0).reset_index()

In [24]:
#年収別_初期交渉設定データ
df1 = shoki_settei.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_below700").fillna(0)
df2 = shoki_settei.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_below700").fillna(0)
below_shoki_settei =  pd.concat([df1,df2],axis=1).reset_index()

df1 = shoki_settei.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_between700and1000").fillna(0)
df2 = shoki_settei.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_between700and1000").fillna(0)
between_shoki_settei =  pd.concat([df1,df2],axis=1).reset_index()

df1 = shoki_settei.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_over1000").fillna(0)
df2 = shoki_settei.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_over1000").fillna(0)
over_shoki_settei =  pd.concat([df1,df2],axis=1).reset_index()

income_shoki_settei_all=  pd.concat([below_shoki_settei,between_shoki_settei,over_shoki_settei],axis=0).reset_index()

#年収別_初期交渉実施データ
df1 = shoki_jisshi.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_below700").fillna(0)
df2 = shoki_jisshi.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_below700").fillna(0)
below_shoki_jisshi =  pd.concat([df1,df2],axis=1).reset_index()

df1 = shoki_jisshi.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_between700and1000").fillna(0)
df2 = shoki_jisshi.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_between700and1000").fillna(0)
between_shoki_jisshi =  pd.concat([df1,df2],axis=1).reset_index()

df1 = shoki_jisshi.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_over1000").fillna(0)
df2 = shoki_jisshi.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_over1000").fillna(0)
over_shoki_jisshi =  pd.concat([df1,df2],axis=1).reset_index()

income_shoki_jisshi_all=  pd.concat([below_shoki_jisshi,between_shoki_jisshi,over_shoki_jisshi],axis=0).reset_index()

#年収別_再交渉設定データ
df1 = shoki_sai_settei.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_below700").fillna(0)
df2 = shoki_sai_settei.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_below700").fillna(0)
below_shoki_sai_settei =  pd.concat([df1,df2],axis=1).reset_index()

df1 = shoki_sai_settei.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_between700and1000").fillna(0)
df2 = shoki_sai_settei.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_between700and1000").fillna(0)
between_shoki_sai_settei =  pd.concat([df1,df2],axis=1).reset_index()

df1 = shoki_sai_settei.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_over1000").fillna(0)
df2 = shoki_sai_settei.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_over1000").fillna(0)
over_shoki_sai_settei =  pd.concat([df1,df2],axis=1).reset_index()

income_shoki_sai_settei_all=  pd.concat([below_shoki_settei,between_shoki_settei,over_shoki_settei],axis=0).reset_index()

#年収別_再交渉実施データ
df1 = shoki_sai_jisshi.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_below700").fillna(0)
df2 = shoki_sai_jisshi.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_below700").fillna(0)
below_shoki_sai_jisshi =  pd.concat([df1,df2],axis=1).reset_index()

df1 = shoki_sai_jisshi.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_between700and1000").fillna(0)
df2 = shoki_sai_jisshi.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_between700and1000").fillna(0)
between_shoki_sai_jisshi =  pd.concat([df1,df2],axis=1).reset_index()

df1 = shoki_sai_jisshi.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_over1000").fillna(0)
df2 = shoki_sai_jisshi.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_over1000").fillna(0)
over_shoki_sai_jisshi =  pd.concat([df1,df2],axis=1).reset_index()

income_shoki_sai_jisshi_all=  pd.concat([below_shoki_jisshi,between_shoki_jisshi,over_shoki_jisshi],axis=0).reset_index()

In [25]:
income_shoki_sai_jisshi_all

,index,type,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,2019/10,2019/11,2019/12,2020/01,2020/02,2020/03,2020/04,2020/05,2020/06,2020/07,2020/08,2020/09,2020/10,2020/11,2020/12,2021/01,2021/02,2021/03,2021/04,2021/05,2021/06,2021/07,2021/08,2021/09,2021/10,2021/11,2021/12,2022/01,2022/02,2022/03,2022/04,2022/05,2022/06,2022/07,2022/08,2022/09,2022/10,2022/11,2022/12,2023/01,2023/02,2023/03,2023/04,2023/05,2023/06,2023/07,2023/08,2023/09
0,0,初期交渉実施数,66,82,98,83,127,110,124,150,79,45,35,51,33,50,37,47,13,37,16,25,33,25,14,38,46,45,9,28,42,43,42,30,38,42,30,48,46,51,60,36,29,32,21,16,10,19,11,9,16,13,16,21,5,9,21,24,10,15,13,8,17,18,19,8
1,0,初期交渉実施数,212,252,412,397,483,346,451,435,176,190,194,256,97,110,187,180,50,93,71,72,85,97,93,126,199,152,114,121,142,184,157,100,101,145,149,149,153,150,153,124,79,57,49,47,59,85,53,60,83,103,66,83,25,35,39,54,29,28,47,60,81,73,72,31
2,0,初期交渉実施数,575,708,863,932,858,815,731,892,576,532,499,642,240,456,536,532,145,236,210,224,229,255,254,275,332,388,270,260,257,310,291,248,269,298,251,215,265,315,277,269,230,206,175,180,151,203,138,126,241,249,197,184,48,68,142,159,119,171,166,153,225,209,204,100


rzshokikosho.replace([np.inf, -np.inf], np.nan, inplace=True)
rzshokikosho.fillna('', inplace=True)

rzshokikosho = rzshokikosho.values.tolist()

#スプレッドシートに記載する
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"C:\Users\suehara\Desktop\お転機BOX\ぱいそん練習\python_ss\credentials.json"
service = ps.get_auth(SCOPES,json_path)
SPREADSHEET_ID = '1ofer5LQADQ9ppKrtQWWbOyhyEmcrBx_c_O4UmVFe3wU'
Sheet_NAME = '初期交渉!A'
Sheet_row = "2"
RANGE_NAME = Sheet_NAME+Sheet_row
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,rzshokikosho,service)

#スプレッドシートに記載する
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"C:\Users\suehara\Desktop\お転機BOX\ぱいそん練習\python_ss\credentials.json"
service = ps.get_auth(SCOPES,json_path)
SPREADSHEET_ID = '1EFxWeFN9b2kWU1qAy98KmjEm9n63-p61HiurqZkEWlQ'
Sheet_NAME = '初期交渉!A'
Sheet_row = "2"
RANGE_NAME = Sheet_NAME+Sheet_row
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,rzshokikosho,service)

## 本交渉データ

In [26]:
honkosho_query = """
SELECT 
  hks.id as honkosho_id,
  hks.anken_id as anken_id,
  an.kohosha_id as kohosha_id,
  khs.tenki_id as tenki_id,
  consts.name as ap_source,
  khs.birth_year,
  khs.annual_income as income,
  format_date('%Y/%m/%d',hks.kosho_setteibi) as setteibi,
  format_date('%Y/%m/%d',hks.kosho_yoteibi) as yoteibi,
  format_date('%Y/%m/%d',hks.kosho_jisshibi) as jisshibi,
  syi1.sei_plus as kohosha_tanto,
  syi2.sei_plus as kigyo_tanto,
  hks.kosho_seq
FROM `temp-380708.live_rhs.honkoshos` hks
left join `temp-for-sandbox.ronzanmi__mart.vw_honkosho_saikoshos` vhon on hks.id = vhon.Honkosho_id
left join `temp-380708.live_rhs.ankens` an on vhon.anken_id = an.id
left join `temp-380708.live_rhs.kohoshas` khs on vhon.kohosha_id = khs.id
left join `temp-380708.live_company.syain` syi1 on hks.kohosha_tanto = syi1.user_id
left join `temp-380708.live_company.syain` syi2 on an.kigyo_tanto = syi2.user_id
left join (SELECT 
              code,
              name
             FROM `temp-380708.live_rhs.sys_consts`
             where group_code = 19) consts on khs.ap_source = consts.code
where hks.kosho_setteibi >= date '2019-10-02'
and consts.name = '転機社長名鑑'
and hks.kosho_seq = 1
order by hks.kosho_setteibi
"""

client = bigquery.Client(credentials=credentials, project=credentials.project_id)
honkosho = client.query(honkosho_query).result().to_dataframe()

honkosho['income_below700'] = honkosho['income'] < 700
honkosho['income_between700and1000'] = (honkosho['income'] >= 700) & (honkosho['income'] < 1000)
honkosho['income_over1000'] = honkosho['income'] >= 1000

hon_settei = honkosho.copy()
hon_jisshi = honkosho.copy()

honkosho

,honkosho_id,anken_id,kohosha_id,tenki_id,ap_source,birth_year,income,setteibi,yoteibi,jisshibi,kohosha_tanto,kigyo_tanto,kosho_seq,income_below700,income_between700and1000,income_over1000
0,6481,6040,13545,<NA>,転機社長名鑑,1961,620.0,2019/10/02,2019/10/11,None,花田和,吉山史,1,True,False,False
1,6478,6037,14340,<NA>,転機社長名鑑,1965,950.0,2019/10/02,2019/10/15,2019/10/15,運上禎,堀汐,1,False,True,False
2,6482,6041,14386,<NA>,転機社長名鑑,1955,4500.0,2019/10/02,2019/10/15,2019/10/15,大塚洋,堀汐,1,False,False,True
3,6483,6042,13339,<NA>,転機社長名鑑,1962,1100.0,2019/10/02,2019/10/08,2019/10/08,小野佑,長松大,1,False,False,True
4,6475,6034,13316,<NA>,転機社長名鑑,1962,1850.0,2019/10/02,2019/10/08,2019/10/08,大谷昌,長松大,1,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4936,17967,17099,16610,49527,転機社長名鑑,1972,750.0,2023/09/12,2023/09/15,None,吉武南,松下将,1,False,True,False
4937,17972,17104,41958,102995,転機社長名鑑,1967,1054.0,2023/09/13,2023/09/22,None,曽根裕,曽根裕,1,False,False,True
4938,17971,17103,46513,108237,転機社長名鑑,1970,800.0,2023/09/13,2023/09/19,None,加藤諒,浅井祐,1,False,True,False
4939,17991,17119,2361,89552,転機社長名鑑,1964,780.0,2023/09/14,2023/09/22,None,原口虎,赤羽勇,1,False,True,False


In [27]:
#本交渉設定テーブルを作成
Q_master5 = Q_master.rename(columns={"toroku_date":"setteibi"})
hon_settei = pd.merge(honkosho,Q_master5,on=("setteibi"),how=("left"))
hon_settei["type"] = "本交渉設定"

#本交渉設定テーブルを作成
Q_master6 = Q_master.rename(columns={"toroku_date":"jisshibi"})
hon_jisshi = pd.merge(honkosho,Q_master6,on=("jisshibi"),how=("left"))
hon_jisshi["type"] = "本交渉実施"

In [28]:
df1 = hon_settei.pivot_table(index=["type"],columns="Q",aggfunc="count",values="setteibi").fillna(0)
df2 = hon_settei.pivot_table(index=["type"],columns="月",aggfunc="count",values="setteibi").fillna(0)
hon_settei_all=  pd.concat([df1,df2],axis=1).reset_index()

df1 = hon_jisshi.pivot_table(index=["type"],columns="Q",aggfunc="count",values="jisshibi").fillna(0)
df2 = hon_jisshi.pivot_table(index=["type"],columns="月",aggfunc="count",values="jisshibi").fillna(0)
hon_jisshi_all=  pd.concat([df1,df2],axis=1).reset_index()

honkosho_all = pd.concat([hon_settei_all,hon_jisshi_all],axis=0).reset_index()

In [29]:
#年収別_初期交渉設定データ
df1 = hon_settei.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_below700").fillna(0)
df2 = hon_settei.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_below700").fillna(0)
below_hon_settei =  pd.concat([df1,df2],axis=1).reset_index()

df1 = hon_settei.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_between700and1000").fillna(0)
df2 = hon_settei.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_between700and1000").fillna(0)
between_hon_settei =  pd.concat([df1,df2],axis=1).reset_index()

df1 = hon_settei.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_over1000").fillna(0)
df2 = hon_settei.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_over1000").fillna(0)
over_hon_settei =  pd.concat([df1,df2],axis=1).reset_index()

income_hon_settei_all=  pd.concat([below_hon_settei,between_hon_settei,over_hon_settei],axis=0).reset_index()

#年収別_初期交渉実施データ
df1 = hon_jisshi.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_below700").fillna(0)
df2 = hon_jisshi.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_below700").fillna(0)
below_hon_jisshi =  pd.concat([df1,df2],axis=1).reset_index()

df1 = hon_jisshi.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_between700and1000").fillna(0)
df2 = hon_jisshi.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_between700and1000").fillna(0)
between_hon_jisshi =  pd.concat([df1,df2],axis=1).reset_index()

df1 = hon_jisshi.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_over1000").fillna(0)
df2 = hon_jisshi.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_over1000").fillna(0)
over_hon_jisshi =  pd.concat([df1,df2],axis=1).reset_index()

income_hon_jisshi_all=  pd.concat([below_hon_jisshi,between_hon_jisshi,over_hon_jisshi],axis=0).reset_index()

## 成約データ

In [30]:
#今Qのポイントデータは最新のヨミ表からとってくる
yomi_nowQ = pd.read_excel(r"\\172.16.0.232\CoffeeCrazy\管理グループ\管理部\ポイント割り振り表\【入力用】ポイント表\26期\26-4\ロンザン統合版ポイント表\【26-4Q】入力用顧客支持ポイント表（ロンザン）.xlsm",header=20,usecols = (range(0, 43)))
yomi_nowQ = yomi_nowQ[:-1]

#RPA事業部のクロスセルは除く(顧問名が「塩澤昌紘」はRPA事業部のクロスセル案件)
yomi_nowQ = yomi_nowQ[~yomi_nowQ['候補者'].isin(['塩澤昌紘'])]
yomi_nowQ = yomi_nowQ[~yomi_nowQ['売上種別（商品内容）'].isin(['RPAコンサル'])]
yomi_nowQ = yomi_nowQ[~yomi_nowQ['売上種別（商品内容）'].isin(['ビジネスタンク'])]
yomi_nowQ.to_excel('yomi_nowQ.xlsx')

yomi_nowQ.head(2)

yomiold = pd.read_excel(path1 + "\yojitu\RZポイント表過去データ（22-3Qまで）最新.xlsx",usecols  =[5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47])

yomiold = yomiold[yomiold['計上日'] >= "2019/10/02"]

seiyaku = pd.concat([yomiold,yomi_nowQ],axis=0,ignore_index=True)

In [31]:
seiyaku

,No,案件id\n（RZ）,計上月,期,月,案件\nNo\n（SC）,入力者,計上日,受注日,クライアント正式名称,候補者,売上種別（商品内容）,差分\n（該当場合のみ）,紹介引当/PM引当\n特殊ポイント\n（該当場合のみ）,基準年収,報酬率,完保\n成約\n割振比,グループ企画料,サービス\n引当\n係数,キャンセル\n引当\n係数,営業売上合計,所属課,氏名,割合,受注額,査定用\n売上,顧客支持ポイント,引き継ぎP,担当,担当\n押印,備考①,備考②,備考③,シニアスカウト\n（ヨミ表と一致）,内定数フラグ,企業アポソース,特殊フラグ,提示/前年度,基準年収.1,報酬率.1,d,d.1,アポソース,紹介引当/PM引当\n特殊顧客支持ポイント\n（該当場合のみ）,ロンザン氏名\n表記揺れチェック
0,NaN,0,10月,23.0,NaN,NaN,糸井,2019-10-07 00:00:00,2019-10-07 00:00:00,株式会社山一地所,安永周平,顧問名鑑,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ロンザン,小橋一,NaN,NaN,NaN,10.96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,0,10月,23.0,NaN,NaN,藤井,2019-10-15 00:00:00,2019-09-30 00:00:00,エターナル,NaN,WEBコンサル,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ロンザン,宮本亜,NaN,NaN,NaN,25.0215,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,0,10月,23.0,NaN,NaN,横須賀,2019-10-23 00:00:00,2019-10-23 00:00:00,ユナイテッド・プレシジョン・テクノロジーズ株式会社(株式会社協成),NaN,採用コンサルティング報酬,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ロンザン,宮本亜,NaN,NaN,NaN,21.42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,0,10月,23.0,NaN,NaN,横須賀,2019-10-23 00:00:00,2019-10-23 00:00:00,ユナイテッド・プレシジョン・テクノロジーズ株式会社(株式会社協成),NaN,採用コンサルティング報酬,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ロンザン,宮本亜,NaN,NaN,NaN,6.12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,0,10月,23.0,NaN,NaN,横須賀,2019-10-25 00:00:00,2019-10-25 00:00:00,無垢スタイル建築設計株式会社,NaN,採用コンサルティング報酬,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ロンザン,宮本亜,NaN,NaN,NaN,24.17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24807,NaN,NaN,9月,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24808,NaN,NaN,9月,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24809,NaN,NaN,9月,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24810,NaN,NaN,9月,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
seiyaku = seiyaku[seiyaku['氏名']=="転機"]
seiyaku = seiyaku[['案件id\n（RZ）','計上日','受注日', 'クライアント正式名称', '候補者', '売上種別（商品内容）', 
            '基準年収', '報酬率', '完保\n成約\n割振比','営業売上合計','氏名','受注額','顧客支持ポイント','内定数フラグ',
            '提示/前年度', '基準年収.1','アポソース']]
seiyaku = seiyaku.rename(columns={"案件id\n（RZ）":"id"})
seiyaku["id"] = seiyaku["id"].fillna(0).astype("int64")

In [33]:
seiyaku["営業売上合計"] = seiyaku["営業売上合計"]*10000
seiyaku["顧客支持ポイント"] = seiyaku["顧客支持ポイント"]*10000

In [34]:
seiyaku

,id,計上日,受注日,クライアント正式名称,候補者,売上種別（商品内容）,基準年収,報酬率,完保\n成約\n割振比,営業売上合計,氏名,受注額,顧客支持ポイント,内定数フラグ,提示/前年度,基準年収.1,アポソース
7,5906,2019-10-25 00:00:00,2019-10-23 00:00:00,アサゴエ工業株式会社,牧原 慎一 氏,シニアスカウト報酬,872.016,0.58,NaN,5057692.8,転機,505.76928,1062115.488,NaN,NaN,NaN,転機
18,5836,2019-10-25 00:00:00,2019-10-17 00:00:00,株式会社光洲産業,池田 達也 氏,シニアスカウト報酬,800.0,0.62,NaN,4960000.0,転機,496,1041600.0,NaN,NaN,NaN,転機
31,5518,2019-10-30 00:00:00,2019-10-17 00:00:00,株式会社アフターフィットエンジニアリング,栗原 聖之 氏,シニアスカウト報酬,3000.0,0.58,NaN,17400000.0,転機,1740,3654000.0,NaN,NaN,NaN,転機
75,6026,2019-10-31 00:00:00,2019-10-30 00:00:00,レック株式会社,松熊 祥子 氏,シニアスカウト報酬,1799.65,0.58,NaN,10437970.0,転機,1043.797,2191973.7,NaN,NaN,NaN,転機
94,6033,2019-10-31 00:00:00,2019-10-31 00:00:00,株式会社ナンブ,伊藤 嘉英 氏,シニアスカウト報酬,850.0,0.6,NaN,5100000.0,転機,510,1071000.0,NaN,NaN,NaN,転機
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22139,15938,2023-07-31 00:00:00,2023-07-31 00:00:00,株式会社ブシロード,藤井 学 氏,シニアスカウト報酬,900,0.62,1,5580000,転機,558,1152549.0,NaN,提示年収,9000000,転機
22177,15843,2023-07-31 00:00:00,2023-07-28 00:00:00,太田油脂株式会社,廣瀬 進 氏,シニアスカウト報酬,1066.4302,0.65,1,6931796.3,転機,693.17963,1431762.525765,NaN,提示年収,10664302,転機
22230,16022,2023-07-31 00:00:00,2023-07-26 00:00:00,ケアサポート株式会社,池野 暢宏 氏,シニアスカウト報酬,971.242,0.67,1,6507321.4,転機,650.73214,1265023.28016,NaN,提示年収,9712420,転機
22405,15781,2023-08-21 00:00:00,2023-07-31 00:00:00,アラインテック株式会社,山崎 裕之 氏,シニアスカウト報酬,1143.9452,0.65,1,7435643.8,転機,743.56438,1445489.15472,NaN,提示年収,11439452,転機


In [35]:
anken_query = """
SELECT
  an.id,
  koho.id,
  koho.seimei,
  koho.annual_income as income
FROM `temp-380708.live_rhs.ankens` an
left join `temp-380708.live_rhs.kohoshas` koho on koho.id = an.kohosha_id
"""

client = bigquery.Client(credentials=credentials, project=credentials.project_id)
anken = client.query(anken_query).result().to_dataframe()

In [36]:
seiyaku = pd.merge(seiyaku,anken,on=("id"),how=("left"))

In [37]:
seiyaku['income_below700'] = seiyaku['income'] < 700
seiyaku['income_between700and1000'] = (seiyaku['income'] >= 700) & (seiyaku['income'] < 1000)
seiyaku['income_over1000'] = seiyaku['income'] >= 1000
seiyaku

,id,計上日,受注日,クライアント正式名称,候補者,売上種別（商品内容）,基準年収,報酬率,完保\n成約\n割振比,営業売上合計,氏名,受注額,顧客支持ポイント,内定数フラグ,提示/前年度,基準年収.1,アポソース,id_1,seimei,income,income_below700,income_between700and1000,income_over1000
0,5906,2019-10-25 00:00:00,2019-10-23 00:00:00,アサゴエ工業株式会社,牧原 慎一 氏,シニアスカウト報酬,872.016,0.58,NaN,5057692.8,転機,505.76928,1062115.488,NaN,NaN,NaN,転機,14151,牧原 慎一,940.0,False,True,False
1,5836,2019-10-25 00:00:00,2019-10-17 00:00:00,株式会社光洲産業,池田 達也 氏,シニアスカウト報酬,800.0,0.62,NaN,4960000.0,転機,496,1041600.0,NaN,NaN,NaN,転機,10835,池田 達也,650.0,True,False,False
2,5518,2019-10-30 00:00:00,2019-10-17 00:00:00,株式会社アフターフィットエンジニアリング,栗原 聖之 氏,シニアスカウト報酬,3000.0,0.58,NaN,17400000.0,転機,1740,3654000.0,NaN,NaN,NaN,転機,11710,栗原 聖之,2300.0,False,False,True
3,6026,2019-10-31 00:00:00,2019-10-30 00:00:00,レック株式会社,松熊 祥子 氏,シニアスカウト報酬,1799.65,0.58,NaN,10437970.0,転機,1043.797,2191973.7,NaN,NaN,NaN,転機,13115,松熊 祥子,1300.0,False,False,True
4,6033,2019-10-31 00:00:00,2019-10-31 00:00:00,株式会社ナンブ,伊藤 嘉英 氏,シニアスカウト報酬,850.0,0.6,NaN,5100000.0,転機,510,1071000.0,NaN,NaN,NaN,転機,7539,I Y,750.0,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
687,15938,2023-07-31 00:00:00,2023-07-31 00:00:00,株式会社ブシロード,藤井 学 氏,シニアスカウト報酬,900,0.62,1,5580000,転機,558,1152549.0,NaN,提示年収,9000000,転機,43848,藤井 学,1000.0,False,False,True
688,15843,2023-07-31 00:00:00,2023-07-28 00:00:00,太田油脂株式会社,廣瀬 進 氏,シニアスカウト報酬,1066.4302,0.65,1,6931796.3,転機,693.17963,1431762.525765,NaN,提示年収,10664302,転機,41925,廣瀬 進,1100.0,False,False,True
689,16022,2023-07-31 00:00:00,2023-07-26 00:00:00,ケアサポート株式会社,池野 暢宏 氏,シニアスカウト報酬,971.242,0.67,1,6507321.4,転機,650.73214,1265023.28016,NaN,提示年収,9712420,転機,17319,池野 暢宏,900.0,False,True,False
690,15781,2023-08-21 00:00:00,2023-07-31 00:00:00,アラインテック株式会社,山崎 裕之 氏,シニアスカウト報酬,1143.9452,0.65,1,7435643.8,転機,743.56438,1445489.15472,NaN,提示年収,11439452,転機,34776,山崎 裕之,1200.0,False,False,True


honkosho['income_below700'] = honkosho['income'] < 700
honkosho['income_between700and1000'] = (honkosho['income'] >= 700) & (honkosho['income'] < 1000)
honkosho['income_over1000'] = honkosho['income'] >= 1000

hon_settei = honkosho.copy()
hon_jisshi = honkosho.copy()

honkosho_query

In [38]:
#成約テーブルを作成
Q_master7 = Q_master.rename(columns={"toroku_date":"計上日"})
seiyaku['計上日'] = pd.to_datetime(seiyaku['計上日'])
seiyaku['計上日'] = seiyaku['計上日'].dt.strftime('%Y/%m/%d')

seiyaku = pd.merge(seiyaku,Q_master7,on=("計上日"),how=("left"))
uriage = seiyaku.copy()
point = seiyaku.copy()
seiyaku["type"] = "成約数"
uriage["type"]  = "営業売上金額" 
point["type"] = "顧客支持ポイント"

In [39]:
df1 = seiyaku.pivot_table(index=["type"],columns="Q",aggfunc="count",values="計上日").fillna(0)
df2 = seiyaku.pivot_table(index=["type"],columns="月",aggfunc="count",values="計上日").fillna(0)
seiyaku_all =  pd.concat([df1,df2],axis=1).reset_index()

df1 = uriage.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="営業売上合計").fillna(0)
df2 = uriage.pivot_table(index=["type"],columns="月",aggfunc="sum",values="営業売上合計").fillna(0)
uriage_all=  pd.concat([df1,df2],axis=1).reset_index()

df1 = point.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="顧客支持ポイント").fillna(0)
df2 = point.pivot_table(index=["type"],columns="月",aggfunc="sum",values="顧客支持ポイント").fillna(0)
pont_all=  pd.concat([df1,df2],axis=1).reset_index() 

In [40]:
uriage_all

,type,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,2019/10,2019/11,2019/12,2020/01,2020/02,2020/03,2020/04,2020/05,2020/06,2020/07,2020/08,2020/09,2020/10,2020/11,2020/12,2021/01,2021/02,2021/03,2021/04,2021/05,2021/06,2021/07,2021/08,2021/09,2021/10,2021/11,2021/12,2022/01,2022/02,2022/03,2022/04,2022/05,2022/06,2022/07,2022/08,2022/09,2022/10,2022/11,2022/12,2023/01,2023/02,2023/03,2023/04,2023/05,2023/06,2023/07,2023/08
0,営業売上金額,4.113750e+08,3.914894e+08,2.969178e+08,4.614533e+08,387603786.9,3.370449e+08,3.496508e+08,2.372632e+08,2.836466e+08,3.407219e+08,1.658603e+08,2.941373e+08,1.820313e+08,162826943.1,2.488815e+08,75825876.42,72419214.8,93136651.22,2.458192e+08,1.063953e+08,1.458318e+08,1.392623e+08,88622456.19,1.109022e+08,97393168.58,1.239154e+08,1.486387e+08,188899162.7,57339946.8,60162777.08,2.701011e+08,51669652.28,1.094617e+08,1.759136e+08,53807921.2,1.141609e+08,1.816820e+08,41263169.0,54596766.47,1.414033e+08,66465451.99,47157507.28,1.700237e+08,48306792.82,123198098.8,1.692171e+08,70737759.7,40570724.01,54551840.98,55440302.34,5.502184e+07,1.836752e+08,19322305.52,55018373.7,107690626.6,53554653.9,73273689.2,35998600.0,67509824.08,50609274.88,1.307624e+08,62189736.62,13636139.8


In [41]:
df1 = seiyaku.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_below700").fillna(0)
df2 = seiyaku.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_below700").fillna(0)
below_seiyaku =  pd.concat([df1,df2],axis=1).reset_index()

df1 = seiyaku.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_between700and1000").fillna(0)
df2 = seiyaku.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_between700and1000").fillna(0)
between_seiyaku =  pd.concat([df1,df2],axis=1).reset_index()

df1 = seiyaku.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="income_over1000").fillna(0)
df2 = seiyaku.pivot_table(index=["type"],columns="月",aggfunc="sum",values="income_over1000").fillna(0)
over_seiyaku =  pd.concat([df1,df2],axis=1).reset_index()

In [42]:
seiyaku_all = seiyaku_all.fillna(0)

# float64型のカラムを選択
float_cols = seiyaku_all.select_dtypes(include=['float64']).columns

# 選択したカラムをint64型に変換
seiyaku_all[float_cols] = seiyaku_all[float_cols].astype('int64')

In [43]:
ronzan_all = pd.concat([joryu,shokikosho_all,honkosho_all,seiyaku_all],axis=0).reset_index()
ronzan_all = ronzan_all.drop(["level_0","index"],axis=1)
ronzan_all

,type,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,2019/10,2019/11,2019/12,2020/01,2020/02,2020/03,2020/04,2020/05,2020/06,2020/07,2020/08,2020/09,2020/10,2020/11,2020/12,2021/01,2021/02,2021/03,2021/04,2021/05,2021/06,2021/07,2021/08,2021/09,2021/10,2021/11,2021/12,2022/01,2022/02,2022/03,2022/04,2022/05,2022/06,2022/07,2022/08,2022/09,2022/10,2022/11,2022/12,2023/01,2023/02,2023/03,2023/04,2023/05,2023/06,2023/07,2023/08,2023/09
0,登録数,3325,3622,4186,4577,3885,3930,3273,4110,3453,3826,3666,3038,1176,2208,2823,2661,1099,1370,989,1213,1286,1136,1194,1411,1748,1790,1199,1275,1409,1226,1327,1239,1359,1255,888,1151,1234,1130,1338,1491,1322,1173,1150,1422,1266,1149,1184,1030,1455,1283,1063,637,118,274,827,773,627,813,908,808,1104,1032,1066,518.0
1,手上げ数,1772,1845,2453,2335,2476,2603,2248,2250,1431,1758,1348,1493,712,1218,1749,1804,578,707,503,691,596,557,669,852,957,1010,624,661,867,837,772,657,682,1264,661,733,854,776,834,581,522,556,438,589,671,493,361,407,572,602,516,362,87,181,461,497,346,378,389,672,707,680,733,352.0
2,初期交渉設定数,971,1157,1504,1562,1573,1460,1469,1525,863,847,843,1022,448,711,918,886,313,373,304,379,384,387,406,494,625,641,443,445,485,564,524,433,426,601,445,456,568,519,527,444,337,316,269,258,308,268,229,224,395,375,321,310,83,119,263,294,172,249,294,251,372,341,341,184.0
3,初期交渉実施数,853,1042,1373,1412,1468,1271,1306,1477,832,767,728,949,370,616,760,759,208,366,297,321,347,377,361,439,577,585,393,409,441,537,490,378,408,485,430,412,464,516,490,429,339,295,245,243,220,307,202,195,340,365,279,288,78,112,202,237,158,214,226,221,323,300,295,139.0
4,本交渉設定,427,380,383,402,405,386,428,314,298,276,205,274,186,149,214,214,160,175,95,172,126,83,122,152,107,150,146,104,147,159,99,124,127,135,155,172,101,122,102,83,86,128,95,125,91,61,69,61,78,103,98,65,71,66,52,64,53,32,81,64,71,74,100,35.0
5,本交渉実施,348,320,322,361,358,318,377,279,261,255,180,247,173,105,196,161,75,159,114,108,124,91,97,117,111,119,118,118,86,163,109,89,109,120,115,149,113,75,111,86,61,118,92,97,98,58,65,46,73,77,94,71,66,57,53,41,39,24,53,78,66,57,81,20.0
6,成約数,61,55,46,76,55,55,52,43,41,51,24,44,27,22,29,11,9,14,38,15,21,19,13,17,16,21,24,31,8,8,39,8,16,31,7,18,27,8,10,25,10,8,23,9,17,25,9,7,8,8,7,29,3,9,15,7,10,5,10,6,13,9,2,NaN


In [44]:
##広告費を取得
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"C:\Users\suehara\Desktop\お転機BOX\ぱいそん練習\python_ss\credentials.json"
service = ps.get_auth(SCOPES,json_path)
SPREADSHEET_ID = '14vXxjcdlyY463QjrtEnCw7pJ0oLStXZD8Q42IdW2Zd4'
Sheet_NAME = '広告費!A'
Sheet_row = ":M"
RANGE_NAME = Sheet_NAME+Sheet_row
adcost = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)

In [45]:
adcost

,date,YSS,YDN,GSN,GDN,Facebook（CA),NETMARKETING,SmartNews,LINE CPC,LINE リタゲ,LINE CPF,Microsoft,Total
1,2019/01/01,"10,507","12,754","23,230","18,619","287,064",,,,0,0,,"¥352,174"
2,2019/01/02,"17,706","17,514","38,203","20,034","302,323",,,,0,0,,"¥395,780"
3,2019/01/03,"24,992","20,083","42,186","19,498","299,953",,,,0,0,,"¥406,712"
4,2019/01/04,"23,508","17,970","61,093","21,830","292,329",,,,0,0,,"¥416,730"
5,2019/01/05,"29,423","15,239","58,310","18,247","299,390",,,,0,0,,"¥420,609"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1734,2023/09/30,,,,,,,,,0,,,¥0
1735,2023/10/01,,,,,,,,,0,,,¥0
1736,2023/10/02,,,,,,,,,0,,,¥0
1737,2023/10/03,,,,,,,,,0,,,¥0


In [46]:
adcost['date'] = pd.to_datetime(adcost['date'])
adcost['date'] = adcost['date'].dt.strftime('%Y/%m/%d')
Q_master = Q_master.rename(columns={"toroku_date": "date"})
adcost = pd.merge(adcost,Q_master,how="left",on=("date"))

In [47]:
adcost = adcost[adcost["date"] >= "2019/10/02"]

In [48]:
# '¥'と','を削除
adcost["Total"] = adcost["Total"].str.replace('¥', '').str.replace(',', '')

# float型に変換
adcost["Total"] = adcost["Total"].astype(float)

# int型に変換
adcost["Total"] = adcost["Total"].astype(int)

In [49]:
adcost["type"] = "広告費"

In [50]:
df1 = adcost.pivot_table(index=["type"],columns="Q",aggfunc="sum",values="Total").fillna(0)
df2 = adcost.pivot_table(index=["type"],columns="月",aggfunc="sum",values="Total").fillna(0)
adcost_all=  pd.concat([df1,df2],axis=1).reset_index()

In [51]:
juyo = pd.concat([adcost_all,uriage_all,pont_all],axis=0).reset_index()

In [52]:
juyo = juyo.drop(["index"],axis=1)

In [53]:
#ロンザン年収帯別データをまとめ
ronzan_below_all = pd.concat([below_toroku,
                              below_teage,
                              below_shoki_settei,
                              below_shoki_jisshi,
                              below_hon_settei,
                              below_hon_jisshi,
                              below_seiyaku],axis=0).reset_index()
ronzan_below_all = ronzan_below_all.drop(["index"],axis=1)

ronzan_between_all = pd.concat([between_toroku,
                                between_teage,
                                between_shoki_settei,
                                between_shoki_jisshi,
                                between_hon_settei,
                                between_hon_jisshi,
                                between_seiyaku],axis=0).reset_index()
ronzan_between_all = ronzan_between_all.drop(["index"],axis=1)

ronzan_over_all = pd.concat([over_toroku,
                             over_teage,
                             over_shoki_settei,
                             over_shoki_jisshi,
                             over_hon_settei,
                             over_hon_jisshi,
                             over_seiyaku],axis=0).reset_index()
ronzan_over_all = ronzan_over_all.drop(["index"],axis=1)

In [54]:
ronzan_below_all

,type,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,2019/10,2019/11,2019/12,2020/01,2020/02,2020/03,2020/04,2020/05,2020/06,2020/07,2020/08,2020/09,2020/10,2020/11,2020/12,2021/01,2021/02,2021/03,2021/04,2021/05,2021/06,2021/07,2021/08,2021/09,2021/10,2021/11,2021/12,2022/01,2022/02,2022/03,2022/04,2022/05,2022/06,2022/07,2022/08,2022/09,2022/10,2022/11,2022/12,2023/01,2023/02,2023/03,2023/04,2023/05,2023/06,2023/07,2023/08,2023/09,22-2Q,2019/01
0,登録数,1383,1396,1799,1913,1712,1699,1370,1624,1324,1256,1322,1046,426,705,964,773,450,573,414,462,501,434,523,602,749,688,534,561,623,484,643,568,615,478,349,525,496,425,549,596,516,478,395,466,422,380,454,348,517,416,378,232,38,79,324,231,187,288,317,288,358,333,288,137.0,NaN,NaN
1,手上げ数,376,331,465,451,528,452,463,391,210,209,146,172,137,126,240,255,133,143,108,132,109,88,144,158,162,216,114,116,166,155,207,146,126,180,139,163,161,137,162,87,75,91,53,68,89,49,47,33,68,54,68,47,14,33,93,79,24,20,28,112,105,96,108,46.0,0.0,0.0
2,初期交渉設定数,69,84,101,90,123,118,130,148,73,47,35,54,40,54,42,47,21,35,17,33,31,18,14,45,43,44,16,27,41,50,32,39,30,49,34,42,54,53,60,34,29,27,19,16,15,16,11,7,17,17,14,22,7,10,25,24,12,17,14,7,20,21,16,10.0,NaN,NaN
3,初期交渉実施数,66,82,98,83,127,110,124,150,79,45,35,51,33,50,37,47,13,37,16,25,33,25,14,38,46,45,9,28,42,43,42,30,38,42,30,48,46,51,60,36,29,32,21,16,10,19,11,9,16,13,16,21,5,9,21,24,10,15,13,8,17,18,19,8.0,NaN,NaN
4,本交渉設定,20,27,25,36,27,26,38,27,23,15,4,20,13,7,10,9,7,8,6,11,10,6,9,5,11,12,10,13,13,6,8,5,10,11,11,14,13,8,8,11,8,7,9,8,5,1,1,0,4,5,9,5,4,2,7,2,1,4,4,4,2,4,4,1.0,NaN,NaN
5,本交渉実施,15,25,17,31,21,19,33,25,19,15,3,18,11,4,8,9,3,7,5,6,12,7,7,2,8,9,11,11,7,5,9,2,6,11,5,12,16,4,9,11,7,6,7,6,8,1,0,1,2,7,5,6,3,1,7,2,1,1,1,5,2,4,5,0.0,NaN,NaN
6,成約数,3,2,4,8,2,3,6,8,4,5,1,4,1,1,2,0,1,1,1,0,1,1,3,1,0,3,3,2,0,0,2,0,1,2,0,2,4,0,2,6,0,2,2,1,2,2,1,0,0,0,0,4,0,0,1,1,0,0,1,0,1,0,0,NaN,NaN,NaN


In [55]:
#スプレッドに転記するための下準備
ronzan_all.replace([np.inf, -np.inf], np.nan, inplace=True)
ronzan_all.fillna('', inplace=True)
ronzan_all = ronzan_all.values.tolist()

juyo.replace([np.inf, -np.inf], np.nan, inplace=True)
juyo.fillna('', inplace=True)
juyo = juyo.values.tolist()

ronzan_below_all.replace([np.inf, -np.inf], np.nan, inplace=True)
ronzan_below_all.fillna('', inplace=True)
ronzan_below_all = ronzan_below_all.values.tolist()

ronzan_between_all.replace([np.inf, -np.inf], np.nan, inplace=True)
ronzan_between_all.fillna('', inplace=True)
ronzan_between_all = ronzan_between_all.values.tolist()

ronzan_over_all.replace([np.inf, -np.inf], np.nan, inplace=True)
ronzan_over_all.fillna('', inplace=True)
ronzan_over_all = ronzan_over_all.values.tolist()

In [56]:
#スプレッドシートに記載する
SPREADSHEET_ID = '1ofer5LQADQ9ppKrtQWWbOyhyEmcrBx_c_O4UmVFe3wU'
Sheet_NAME = '【RZ】年収帯別KPI!A'
Sheet_row = "2"
RANGE_NAME = Sheet_NAME+Sheet_row
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,ronzan_all,service)

SPREADSHEET_ID = '1ofer5LQADQ9ppKrtQWWbOyhyEmcrBx_c_O4UmVFe3wU'
Sheet_NAME = '【RZ】年収帯別KPI!A'
Sheet_row = "10"
RANGE_NAME = Sheet_NAME+Sheet_row
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,juyo,service)

SPREADSHEET_ID = '1ofer5LQADQ9ppKrtQWWbOyhyEmcrBx_c_O4UmVFe3wU'
Sheet_NAME = '【RZ】年収帯別KPI!A'
Sheet_row = "25"
RANGE_NAME = Sheet_NAME+Sheet_row
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,ronzan_below_all,service)

SPREADSHEET_ID = '1ofer5LQADQ9ppKrtQWWbOyhyEmcrBx_c_O4UmVFe3wU'
Sheet_NAME = '【RZ】年収帯別KPI!A'
Sheet_row = "42"
RANGE_NAME = Sheet_NAME+Sheet_row
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,ronzan_between_all,service)

SPREADSHEET_ID = '1ofer5LQADQ9ppKrtQWWbOyhyEmcrBx_c_O4UmVFe3wU'
Sheet_NAME = '【RZ】年収帯別KPI!A'
Sheet_row = "59"
RANGE_NAME = Sheet_NAME+Sheet_row
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,ronzan_over_all,service)

## レイノスKPI

In [60]:
scteage = teageinfo.query('フリ先 == "レイノス"')
scteage = scteage[["ID","登録日時","フリ先","手あげ日付","担当"]]

In [61]:
scteage

,ID,登録日時,フリ先,手あげ日付,担当
45,167,2016/04/16,レイノス,2016/04/16,
48,172,2016/04/25,レイノス,2016/04/25,
51,175,2016/04/26,レイノス,2016/04/26,
52,176,2016/04/27,レイノス,2016/04/27,
61,187,2016/04/28,レイノス,2016/04/28,
...,...,...,...,...,...
50222,108986,2023/09/11,レイノス,2023/09/12,藤下輝幸
50228,108992,2023/09/11,レイノス,2023/09/13,上大園剛
50234,108998,2023/09/12,レイノス,2023/09/13,小高拓
50237,109001,2023/09/12,レイノス,2023/09/15,小高拓


In [ ]:
""

In [ ]:
schonkosho_query = """
SELECT
  rc.tenki_id,
	can.ap_kakutoku as shokikosho_setteibi,
	rc.first_nmen_date as first_jisshi_date,
	can.nmen_zisshi as shokikosho_jisshibi,
	CASE
		WHEN com.settei8 is not null THEN com.settei8 
		WHEN com.settei7 is not null THEN com.settei7
		WHEN com.settei6 is not null THEN com.settei6 
		WHEN com.settei5 is not null THEN com.settei5 
		WHEN com.settei4 is not null THEN com.settei4 
		WHEN com.settei3 is not null THEN com.settei3 
		WHEN com.settei2 is not null THEN com.settei2
		WHEN com.settei1 is not null THEN com.settei1 ELSE null
	END as honkosho_setteibi,
	CASE
		WHEN com.zisshi8 is not null THEN com.zisshi8
		WHEN com.zisshi7 is not null THEN com.zisshi7 
		WHEN com.zisshi6 is not null THEN com.zisshi6 
		WHEN com.zisshi5 is not null THEN com.zisshi5 
		WHEN com.zisshi4 is not null THEN com.zisshi4 
		WHEN com.zisshi3 is not null THEN com.zisshi3 
		WHEN com.zisshi2 is not null THEN com.zisshi2
		WHEN com.zisshi1 is not null THEN com.zisshi1 ELSE null
	END as honkosho_jissibi,
	com.seiyaku,
	can.kouhosya_tantou as kohosha_tanto,
	can.client_tantou as kigyo_tanto,
	can.mendan_tantou as mendan_tanto,
	CASE
		WHEN can.zisshi LIKE 'キャンセル%' THEN 'CXL' 
		WHEN can.zisshi LIKE '%実施%' THEN '実施' ELSE '-' 
	END AS CXL
FROM `temp-380708.live_sugarcrm52.rc_race_candidate_management` rc
left join `temp-380708.live_saltcrm.candidate` can on can.sugarid = rc.candidate_no
left join `temp-380708.live_saltcrm.company` com on com.candidate_id = can.id
where rc.tenki_id is not null 
AND rc.deleted = 0
and can.ap_kakutoku >= date '2019-10-01'
order by can.ap_kakutoku
"""

client = bigquery.Client(credentials=credentials, project=credentials.project_id)
schonkosho = client.query(schonkosho_query).result().to_dataframe()

In [ ]:
schonkosho["tenki_id"] = schonkosho["tenki_id"].astype(str)

In [ ]:
schonkosho = pd.merge(scteage,schonkosho,how="right",on=("tenki_id"))

In [ ]:
schonkosho = schonkosho.query('フリ先 == "レイノス"')

In [ ]:
schonkosho

In [ ]:
schonkosho["first_jisshi_date"] = schonkosho["first_jisshi_date"].astype(str)
schonkosho["shokikosho_setteibi"] = schonkosho["shokikosho_setteibi"].astype(str)
schonkosho["shokikosho_jisshibi"] = schonkosho["shokikosho_jisshibi"].astype(str)
schonkosho["honkosho_setteibi"] = schonkosho["honkosho_setteibi"].astype(str)
schonkosho["honkosho_jissibi"] = schonkosho["honkosho_jissibi"].astype(str)
schonkosho["seiyaku"] = schonkosho["seiyaku"].astype(str)

schonkosho["first_jisshi_date"] = schonkosho["first_jisshi_date"].str.replace('NaT', '')
schonkosho["shokikosho_setteibi"] = schonkosho["shokikosho_setteibi"].str.replace('NaT', '')
schonkosho["shokikosho_jisshibi"] = schonkosho["shokikosho_jisshibi"].str.replace('NaT', '')
schonkosho["honkosho_setteibi"] = schonkosho["honkosho_setteibi"].str.replace('NaT', '')
schonkosho["honkosho_jissibi"] = schonkosho["honkosho_jissibi"].str.replace('NaT', '')
schonkosho["seiyaku"] = schonkosho["seiyaku"].str.replace('NaT', '')

In [ ]:
schonkosho.replace([np.inf, -np.inf], np.nan, inplace=True)
schonkosho.fillna('', inplace=True)

schonkosho = schonkosho.values.tolist()

In [ ]:
#スプレッドシートに記載する
SPREADSHEET_ID = '1ofer5LQADQ9ppKrtQWWbOyhyEmcrBx_c_O4UmVFe3wU'
Sheet_NAME = 'SCKPI!A'
Sheet_row = "2"
RANGE_NAME = Sheet_NAME+Sheet_row
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,schonkosho,service)